In [ ]:
# Notebook: Bode Plot Construction for H(s) = -100s / (s^3 + 9s^2 + 24s + 20)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import matplotlib.pyplot as plt
import control as ct

# 1. Define Transfer Function H(s)
# H(s) = -100s / (s^3 + 9s^2 + 24s + 20)
num_coeffs = [-100.0, 0.0]
den_coeffs = [1.0, 9.0, 24.0, 20.0]
H = ct.tf(num_coeffs, den_coeffs)

# 2. Frequency range spanning from 0.01 to 1000 rad/s (logarithmic scale)
omega = np.logspace(-2, 3, 500) # rad/s

# --- Analytical / Asymptotic Calculations (from textbook solution) ---
# Constant term K = -5 -> Magnitude in dB = 20*log10(5) = 13.9794 dB, Phase = 180°
mag_const_db = 20 * np.log10(5.0) * np.ones_like(omega)
phase_const_deg = 180.0 * np.ones_like(omega)

# Zero at origin: H_z0(j\omega) = j\omega -> Mag = 20*log10(\omega), Phase = 90°
mag_zero_exact_db = 20 * np.log10(omega)
phase_zero_deg = 90.0 * np.ones_like(omega)

# Double pole at alpha1 = 2 (s = -2)
alpha1 = 2.0
mag_pole2_exact_db = -40 * np.log10(np.sqrt(1.0 + (omega / alpha1)**2))
phase_pole2_exact_deg = -2.0 * np.rad2deg(np.arctan(omega / alpha1))

# Simple pole at alpha2 = 5 (s = -5)
alpha2 = 5.0
mag_pole1_exact_db = -20 * np.log10(np.sqrt(1.0 + (omega / alpha2)**2))
phase_pole1_exact_deg = -np.rad2deg(np.arctan(omega / alpha2))

# Total Exact Analytical Superposition
mag_total_exact_db = mag_const_db + mag_zero_exact_db + mag_pole2_exact_db + mag_pole1_exact_db
phase_total_exact_deg = phase_const_deg + phase_zero_deg + phase_pole2_exact_deg + phase_pole1_exact_deg

# 3. Using Python Control Library frequency response method (Warning-free & Unwrapped Phase)
response = H.frequency_response(omega)
mag_ct_db = 20 * np.log10(response.magnitude)
phase_ct_deg = np.rad2deg(np.unwrap(response.phase))

# 4. Plotting using Matplotlib (Subplots for Magnitude and Phase)
fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)

# --- Magnitude Plot ---
ax_mag.semilogx(omega, mag_total_exact_db, 'b-', linewidth=2, label='Exact Curve (Analytical Superposition)')
ax_mag.semilogx(omega, mag_ct_db, 'g:', linewidth=2, label='Control Library Response')
ax_mag.axvline(alpha1, color='gray', linestyle=':', alpha=0.7, label=f'Corner Frequency $\\omega_1 = {alpha1}$ rad/s (Double Pole)')
ax_mag.axvline(alpha2, color='orange', linestyle=':', alpha=0.7, label=f'Corner Frequency $\\omega_2 = {alpha2}$ rad/s (Simple Pole)')
ax_mag.set_ylabel('Magnitude [dB]', fontsize=11)
ax_mag.set_title(r'Bode Diagram of $\mathcal{H}(s) = \frac{-100s}{s^3+9s^2+24s+20}$', fontsize=12)
ax_mag.grid(True, which="both", linestyle=":", alpha=0.6)
ax_mag.legend(loc='lower left', fontsize=9, frameon=True)

# --- Phase Plot ---
ax_phase.semilogx(omega, phase_total_exact_deg, 'b-', linewidth=2, label='Exact Curve (Analytical Superposition)')
ax_phase.semilogx(omega, phase_ct_deg, 'g:', linewidth=2, label='Control Library Response')
ax_phase.axvline(alpha1, color='gray', linestyle=':', alpha=0.7)
ax_phase.axvline(alpha2, color='orange', linestyle=':', alpha=0.7)
ax_phase.set_ylabel('Phase [degrees]', fontsize=11)
ax_phase.set_xlabel(r'Frequency $\omega$ [rad/s]', fontsize=11)
ax_phase.grid(True, which="both", linestyle=":", alpha=0.6)
ax_phase.legend(loc='lower left', fontsize=9, frameon=True)

plt.tight_layout()
plt.show()

# NOTE ON PHASE SHIFT:
# The constant 360° phase offset between the analytical derivation and the control library response 
# is mathematically sound and expected in control systems theory. It arises from the specific choice 
# of branch cut and definition of the negative sign contribution (+180° vs -180°) in complex phase 
# calculations, leaving the physical system dynamics completely unaffected.